In [17]:
import vitaldb

TRACKS = [
    "SNUADC/ECG_II",
    "SNUADC/PLETH",
    "Solar8000/HR",
    "Solar8000/PLETH_SPO2",
    "Solar8000/ART_MBP"
]

caseids = vitaldb.find_cases(TRACKS)

# convert set -> sorted list
caseids = sorted(list(caseids))

print("Number of cases:", len(caseids))
print("First 10 case IDs:")
print(caseids[:10])

/tmp/ipykernel_522676/802679423.py:11: DeprecationWarning: find_cases() relies on the per-track 'trks' index, which is deprecated and may be removed in a future release.
  caseids = vitaldb.find_cases(TRACKS)


Number of cases: 3506
First 10 case IDs:
[1, 3, 4, 7, 10, 13, 14, 16, 17, 19]


In [24]:
import vitaldb
import pandas as pd

caseid = caseids[45]

arr = vitaldb.load_case(
    caseid,
    TRACKS,
    interval=1
)

df = pd.DataFrame(arr, columns=TRACKS)

print(df.shape)
print(df.head())

(19386, 5)
   SNUADC/ECG_II  SNUADC/PLETH  Solar8000/HR  Solar8000/PLETH_SPO2  \
0            NaN           NaN           NaN                   NaN   
1            NaN           NaN           NaN                   NaN   
2      -0.374460     62.833252           NaN                   NaN   
3      -0.295463     35.579437          74.0                  99.0   
4      -0.018975     30.049667           NaN                   NaN   

   Solar8000/ART_MBP  
0                NaN  
1                NaN  
2                NaN  
3                0.0  
4                NaN  


In [25]:
duration_minutes = len(df) / 60

print("Duration (minutes):", duration_minutes)

Duration (minutes): 323.1


In [26]:
ASSESSMENT_MINUTES = 15
PREDICTION_MINUTES = 90

ASSESSMENT_SECONDS = ASSESSMENT_MINUTES * 60
PREDICTION_SECONDS = PREDICTION_MINUTES * 60

# -------------------------
# Check duration
# -------------------------

MIN_REQUIRED = (
    ASSESSMENT_SECONDS +
    PREDICTION_SECONDS
)

if len(df) < MIN_REQUIRED:
    raise ValueError(
        f"Case too short. "
        f"Need {MIN_REQUIRED} seconds, "
        f"got {len(df)}"
    )

# -------------------------
# Split windows
# -------------------------

assessment_df = df.iloc[
    :ASSESSMENT_SECONDS
].copy()

prediction_df = df.iloc[
    ASSESSMENT_SECONDS:
    ASSESSMENT_SECONDS + PREDICTION_SECONDS
].copy()

# -------------------------
# Verify
# -------------------------

print("Assessment shape:")
print(assessment_df.shape)

print()

print("Prediction shape:")
print(prediction_df.shape)

Assessment shape:
(900, 5)

Prediction shape:
(5400, 5)


In [27]:
hr = prediction_df["Solar8000/HR"].dropna()
spo2 = prediction_df["Solar8000/PLETH_SPO2"].dropna()
map_ = prediction_df["Solar8000/ART_MBP"].dropna()

tachycardia = int(hr.max() > 110)

hypotension = int(map_.min() < 65)

hypoxia = int(spo2.min() < 90)

print({
    "tachycardia": tachycardia,
    "hypotension": hypotension,
    "hypoxia": hypoxia
})

{'tachycardia': 0, 'hypotension': 1, 'hypoxia': 0}


In [28]:
print("Max HR:")
print(prediction_df["Solar8000/HR"].max())

print("\nMin MAP:")
print(prediction_df["Solar8000/ART_MBP"].min())

print("\nMin SpO2:")
print(prediction_df["Solar8000/PLETH_SPO2"].min())

Max HR:
88.0

Min MAP:
0.0

Min SpO2:
97.0


In [29]:
INPUT_MINUTES = 15
PREDICTION_MINUTES = 30
STRIDE_MINUTES = 5

INPUT_SECONDS = INPUT_MINUTES * 60
PREDICTION_SECONDS = PREDICTION_MINUTES * 60
STRIDE_SECONDS = STRIDE_MINUTES * 60

In [30]:
df["Solar8000/ART_MBP"] = (
    df["Solar8000/ART_MBP"]
    .mask(df["Solar8000/ART_MBP"] <= 0)
)

df["Solar8000/HR"] = (
    df["Solar8000/HR"]
    .mask(df["Solar8000/HR"] <= 0)
)

df["Solar8000/PLETH_SPO2"] = (
    df["Solar8000/PLETH_SPO2"]
    .mask(df["Solar8000/PLETH_SPO2"] <= 0)
)

In [31]:
def generate_windows(
    df,
    input_seconds=900,
    prediction_seconds=1800,
    stride_seconds=300
):

    windows = []

    total_needed = (
        input_seconds +
        prediction_seconds
    )

    for start in range(
        0,
        len(df) - total_needed + 1,
        stride_seconds
    ):

        input_start = start
        input_end = start + input_seconds

        pred_start = input_end
        pred_end = pred_start + prediction_seconds

        input_df = df.iloc[
            input_start:input_end
        ]

        prediction_df = df.iloc[
            pred_start:pred_end
        ]

        windows.append(
            (
                input_df,
                prediction_df
            )
        )

    return windows

In [32]:
def create_labels(prediction_df):

    hr = prediction_df[
        "Solar8000/HR"
    ].dropna()

    map_ = prediction_df[
        "Solar8000/ART_MBP"
    ].dropna()

    spo2 = prediction_df[
        "Solar8000/PLETH_SPO2"
    ].dropna()

    tachycardia = int(
        len(hr) > 0 and
        hr.max() > 110
    )

    hypotension = int(
        len(map_) > 0 and
        map_.min() < 65
    )

    hypoxia = int(
        len(spo2) > 0 and
        spo2.min() < 90
    )

    return {
        "tachycardia": tachycardia,
        "hypotension": hypotension,
        "hypoxia": hypoxia
    }

In [33]:
rows = []

windows = generate_windows(df)

for idx, (input_df, prediction_df) in enumerate(windows):

    labels = create_labels(
        prediction_df
    )

    rows.append({
        "caseid": caseid,
        "window_id": idx,
        "start_second": idx * 300,

        "tachycardia":
            labels["tachycardia"],

        "hypotension":
            labels["hypotension"],

        "hypoxia":
            labels["hypoxia"]
    })

In [ ]:
all_rows = []

for caseid in caseids:

    try:

        arr = vitaldb.load_case(
            caseid,
            TRACKS,
            interval=1
        )

        df = pd.DataFrame(
            arr,
            columns=TRACKS
        )

        windows = generate_windows(df)

        for idx, (
            input_df,
            prediction_df
        ) in enumerate(windows):

            labels = create_labels(
                prediction_df
            )

            all_rows.append({
                "caseid": caseid,
                "window_id": idx,

                "tachycardia":
                    labels["tachycardia"],

                "hypotension":
                    labels["hypotension"],

                "hypoxia":
                    labels["hypoxia"]
            })

    except Exception:
        pass

In [ ]:
dataset = pd.DataFrame(all_rows)

dataset.to_csv(
    "window_dataset.csv",
    index=False
)

print(dataset.shape)
print(dataset.head())